# Jev — Run the model on the current dataset
No human labels, pilot verification, or holdout are required. Run sections 1–3 to prepare, then section 4 to send requests. The previous evaluation workflow is in `jev_evaluation_reference.ipynb`.


## 1 — Check Python and install (no inference)
Select a **Python 3.10 or newer** notebook kernel. The TypeSafe SDK requires it. Run this cell once; restart the kernel if prompted after installation.

In [4]:
import sys
assert sys.version_info >= (3, 10), 'Select a Python 3.10+ kernel before installing typesafe-sdk.'
%pip install -U typesafe-sdk pandas pydantic python-dotenv openai


Note: you may need to restart the kernel to use updated packages.


## 2 — Settings
Store `TYPESAFE_API_KEY` in `src/.env`. Use a new experiment version if the source, model, or questions change after starting.


In [5]:
import os, json, sys
from pathlib import Path
from datetime import datetime, timezone
import pandas as pd
from dotenv import load_dotenv

ROOT = Path('/Users/rejaimes/repos/coffee-project')
DATA = ROOT / 'data' / 'processed'
OUT = DATA / 'jev'
OUT.mkdir(exist_ok=True)
sys.path.insert(0, str(ROOT / 'src'))
import importlib
import labeling_runner, jev_evaluation
importlib.reload(labeling_runner)
importlib.reload(jev_evaluation)
from jev_evaluation import spec_check, contract_for, predictions, run_jev, timing_cost

load_dotenv(ROOT / 'src' / '.env')
load_dotenv(DATA / '.env')
MODEL = 'jev-1.14.0'
EXPERIMENT_VERSION = 'jev_full_v1'
RESULTS = OUT / f'{EXPERIMENT_VERSION}.jsonl'
RUN_LIMIT = None  # All pending reviews; use an integer to limit this session.
MIN_REQUEST_INTERVAL = 0.5
specs = json.loads((ROOT / 'src' / 'jev_questions_v1.json').read_text())
spec_check(specs)
print('Questions:', list(specs))
print('Results:', RESULTS)


Questions: ['good_wifi', 'easy_parking', 'quiet_atmosphere', 'enough_seating', 'outlets_available', 'work_friendly']
Results: /Users/rejaimes/repos/coffee-project/data/processed/jev/jev_full_v1.jsonl


## 3 — Load current reviews
Load the current dataset directly. Saved results are checked for compatibility so reruns can resume.


In [6]:
source = pd.read_csv(DATA / 'santa_barbara_data.csv', low_memory=False)
reviews = source[['review_id', 'business_id', 'text']].copy()
assert not reviews.empty, 'Source is empty.'
assert reviews.review_id.is_unique, 'Source review IDs must be unique.'
for column in reviews.columns:
    assert reviews[column].map(lambda value: isinstance(value, str) and bool(value.strip())).all(), f'Invalid {column} in source.'
run_contract = contract_for(reviews, specs, MODEL, EXPERIMENT_VERSION, 'full')
saved_rows = predictions(RESULTS, run_contract)
print(f'{len(reviews):,} reviews; {len(saved_rows):,} already saved; {len(reviews) - len(saved_rows):,} pending.')


22,704 reviews; 0 already saved; 22,704 pending.


## 4 — Run Jev (paid API requests)
Execute this cell to start. Successes are saved immediately; reruns skip saved reviews. Interrupt to stop. `RUN_LIMIT = None` processes all pending reviews.


In [7]:
assert os.getenv('TYPESAFE_API_KEY'), 'Add TYPESAFE_API_KEY to src/.env and rerun section 2.'
from typesafe_sdk import TypeSafeClient, RetryPolicy
with TypeSafeClient(model=MODEL, retry=RetryPolicy(max_retries=0), timeout=60.0) as client:
    model_rows = run_jev(
        reviews, specs, client, MODEL, EXPERIMENT_VERSION, RESULTS,
        stage='full', min_interval=MIN_REQUEST_INTERVAL, limit=RUN_LIMIT,
    )


full: 0/22704 already saved; 22704 requests pending.
full: 100/22704 saved; this session failures=0; elapsed=0.8 min
full: 200/22704 saved; this session failures=0; elapsed=1.7 min
full: 300/22704 saved; this session failures=0; elapsed=2.5 min
full: 400/22704 saved; this session failures=0; elapsed=3.4 min
full: 500/22704 saved; this session failures=0; elapsed=4.2 min
full: 600/22704 saved; this session failures=0; elapsed=5.0 min
full: 700/22704 saved; this session failures=0; elapsed=5.9 min
full: 800/22704 saved; this session failures=0; elapsed=6.7 min
full: 900/22704 saved; this session failures=0; elapsed=7.6 min
full: 1000/22704 saved; this session failures=0; elapsed=8.4 min
full: 1100/22704 saved; this session failures=0; elapsed=9.3 min
full: 1200/22704 saved; this session failures=0; elapsed=10.1 min
full: 1300/22704 saved; this session failures=0; elapsed=10.9 min
full: 1400/22704 saved; this session failures=0; elapsed=11.8 min
full: 1500/22704 saved; this session failur

## 5 — Export saved labels and timing
Exports completed reviews, including after a partial run. No human-label evaluation is required.


In [8]:
model_rows = predictions(RESULTS, run_contract)
if model_rows:
    labels = pd.DataFrame([{'review_id': row['review_id'], **row['labels']} for row in model_rows])
    labeled_reviews = reviews.merge(labels, on='review_id', how='inner', validate='one_to_one')
    export_path = RESULTS.with_suffix('.csv')
    labeled_reviews.to_csv(export_path, index=False)
    print(f'Exported {len(labeled_reviews):,}/{len(reviews):,} reviews to {export_path}')
    display(labeled_reviews.head())
else:
    print('No predictions saved yet.')
print(timing_cost(RESULTS, input_price=0.042, output_price=0))


Exported 22,703/22,704 reviews to /Users/rejaimes/repos/coffee-project/data/processed/jev/jev_full_v1.csv


,review_id,business_id,text,good_wifi,easy_parking,quiet_atmosphere,enough_seating,outlets_available,work_friendly
0,L0jv8c2FbpWSlfNC6bbUEA,IDtLPgUrqorrpqSLdfMhZQ,What a great addition to the Funk Zone! Grab ...,0,0,0,0,0,0
1,Gczff1Qxg4WVmlBLPrZwzQ,34Eqv8jXgxg_EEwcsNgeeg,We absolutely love everything we have tried he...,0,0,0,0,0,0
2,bi6GaeWDGceGv62lXTIKQA,bjhCtlYHrkgA5Ku8l-rB3g,Very disappointed. We went to eat at 2:15 on ...,0,0,0,0,0,0
3,H4mSxGeKTv4VFcc9YUmtpQ,-kY_HDP7IMvGl-kBIZVU4A,The espresso here is excellent. My favorite pl...,1,0,1,0,-1,0
4,LFpaQzYkP5Pzm5lEjJpTRw,IDtLPgUrqorrpqSLdfMhZQ,Possibly the best breakfast sandwich EVER. On...,0,0,0,0,0,0


{'successful': 22703, 'failed_attempts': 1, 'wall_seconds_recorded': 12240.98470816703, 'median_request_seconds': np.float64(0.18692264595301822), 'p95_request_seconds': np.float64(0.5267381043988274), 'successful_reviews_per_minute': 111.2802631875825, 'input_tokens_known': 30877355, 'output_tokens_known': 5421689, 'estimated_usd_known_usage': 1.2968489100000002, 'attempts_without_usage': 1, 'input_usd_per_million': 0.042, 'output_usd_per_million': 0}
